In [1]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
import numpy as np
import json
from captum.attr import IntegratedGradients


c:\Users\kz\dsa-proj\dsa4266\network-traffic-classification\cuda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 4266
DATA_DIR = "../../data/processed"

INPUT_DIM = 59
OUTPUT_DIM = 5


N_STEPS = 100

In [3]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [4]:
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim, dropout):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.layer_dim  = layer_dim
        self.lstm = nn.LSTM(input_dim, hidden_dim, layer_dim,
                            batch_first=True,
                            dropout=dropout if layer_dim > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.attn    = nn.Linear(hidden_dim, 1)
        self.fc      = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        #we reset the hidden state for each batch of sequences so that they are independent of each other. This is important for training on batches of sequences.
        out, _ = self.lstm(x)                       # (B, T, H) — zeros state by default
        # attention pooling over time
        scores  = self.attn(out).squeeze(-1)         # (B, T)
        #weight of each timestep
        weights = torch.softmax(scores, dim=1)       # (B, T)
        #weighted average
        pooled  = (out * weights.unsqueeze(-1)).sum(1)  # (B, H)
        #regularisation
        pooled  = self.dropout(pooled)
        return self.fc(pooled)


def load_baseline():
    hidden_dim = 128
    layer_dim = 2
    dropout = 0.2
    lr = 1e-4
    weight_decay = 1e-4
    epochs = 30
    seed = SEED
    patience = 5
    optimizer_name = "adam"
    batch_size = 256
    min_delta = 0

    #set seed for reproducibility
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)


    #create LSTM
    model = LSTMModel(INPUT_DIM, hidden_dim, layer_dim, OUTPUT_DIM, dropout).to(device)
    model.load_state_dict(torch.load("lstm_model.pt", map_location=device))##, weights_only=True))
    #disable dropout
    model.eval()
    return model




In [5]:
model = load_baseline()


In [ ]:

# feats
X_test_raw  = np.load(f"{DATA_DIR}/X_test.npy")


# labels
y_test_raw  = np.load(f"{DATA_DIR}/y_test.npy")

X_test = torch.tensor(X_test_raw, dtype=torch.float32).to(device)
Y_test = torch.tensor(y_test_raw, dtype=torch.long)
y_test_t = torch.tensor(np.argmax(y_test_raw, axis=1), dtype=torch.long).to(device)

#Captum requires gradients to be tracked


In [7]:
def ig_attributions_lstm(model, X_test, y_test, idx, n_steps=N_STEPS,
                         chunk=64, return_delta=False):

    model.eval()
    ig = IntegratedGradients(model)

    attr_list, y_list, pred_list, delta_list = [], [], [], []

    for i in range(0, len(idx), chunk):
        j = idx[i:i + chunk]
        x = X_test[j].to(device).requires_grad_()   # (B, T, F) — real samples
        y = y_test[j].to(device)

        with torch.no_grad():
            pred = model(x).argmax(dim=1)           # (B,)

        out = ig.attribute(
            inputs=x,
            baselines=torch.zeros_like(x),
            target=pred,
            n_steps=n_steps,
            internal_batch_size=256,
            return_convergence_delta=return_delta,
        )
        if return_delta:
            a_x, delta = out
            delta_list.append(delta.detach().cpu())
        else:
            a_x = out

        attr_list.append(a_x.detach().cpu())
        y_list.append(y.cpu())
        pred_list.append(pred.cpu())

    attr        = torch.cat(attr_list,  dim=0)      # (N, T, F)
    true_labels = torch.cat(y_list,     dim=0)      # (N,)
    pred_labels = torch.cat(pred_list,  dim=0)      # (N,)

    if return_delta:
        return attr, true_labels, pred_labels, torch.cat(delta_list, dim=0)
    return attr, true_labels, pred_labels

In [8]:
def balanced_idx(y, per_class=200, seed=42):
    rng = np.random.default_rng(seed)
    idx = [rng.choice(np.where(y == c)[0], min(per_class, (y == c).sum()), replace=False)
           for c in np.unique(y)]
    return torch.tensor(np.concatenate(idx))

idx = balanced_idx(y_test_t.numpy(), per_class=200)
attr, y_true, y_pred, delta = ig_attributions_lstm(model, X_test, y_test=y_test_t, idx=idx, n_steps=N_STEPS, chunk=64, return_delta=True)

In [9]:
def calculate_importance_lstm(attr):
    abs_attr  = attr.abs()
    feat_imp  = abs_attr.sum(dim=1)        # collapse time
    time_imp  = abs_attr.sum(dim=2)        # collapse features
    total_imp = abs_attr.sum(dim=(1, 2))   # collapse both
    return feat_imp, time_imp, total_imp


feat_importance, time_importance, total_importance = calculate_importance_lstm(attr)

In [10]:
print(f"feat_importance: {feat_importance.shape}")
print(f"time_importance: {time_importance.shape}")
print(f"total_importance: {total_importance.shape}")

feat_importance: torch.Size([412, 59])
time_importance: torch.Size([412, 5])
total_importance: torch.Size([412])


In [11]:
with open("../../data/processed/feature_cols.json") as f:
    feature_info = json.load(f)

feature_names = (feature_info["numeric"]+ feature_info["categorical"])


In [12]:
y_true = y_true.detach().cpu()
y_pred = y_pred.detach().cpu()


In [13]:
feat_df = (
    pd.DataFrame({
        "feature":            feature_names,                                   # 59
        "feature_importance": feat_importance.detach().mean(0).cpu().numpy(),  # 59
    })
    .sort_values("feature_importance", ascending=False)
    .reset_index(drop=True)
)
feat_df["share"] = feat_df["feature_importance"] / feat_df["feature_importance"].sum()
feat_df["rank"]  = feat_df["feature_importance"].rank(ascending=False).astype(int)
display(feat_df.head(20))

,feature,feature_importance,share,rank
0,tot_load,1.162645,0.120999,1
1,idle_mean,0.828473,0.086221,2
2,src_ttl_mean,0.716679,0.074586,3
3,num_dst_ports,0.614734,0.063976,4
4,src_rate,0.589653,0.061366,5
5,src_loss,0.550271,0.057268,6
6,run_time_sum,0.468033,0.048709,7
7,src_load,0.467777,0.048682,8
8,tot_rate,0.457457,0.047608,9
9,dst_ttl_mean,0.439863,0.045777,10


In [14]:
time_df = (
    pd.DataFrame({
        "timestep":       [f"t{i}" for i in range(time_importance.shape[1])],  # 5
        "time_importance": time_importance.detach().mean(0).cpu().numpy(),      # 5
    })
    .sort_values("time_importance", ascending=False)
    .reset_index(drop=True)
)
time_df["share"] = time_df["time_importance"] / time_df["time_importance"].sum()
display(time_df)

,timestep,time_importance,share
0,t0,2.899333,0.301739
1,t1,2.506720,0.260879
2,t2,1.973753,0.205412
3,t3,1.344141,0.139887
4,t4,0.884801,0.092083


In [15]:
sample_df = pd.DataFrame({
    "y_true":           y_true.detach().cpu().numpy(),
    "y_pred":           y_pred.detach().cpu().numpy(),
    "total_importance": total_importance.detach().cpu().numpy(),   # 1 per sample
})
sample_df["correct"] = sample_df["y_true"] == sample_df["y_pred"]

print("total attribution per sample — summary")
print(sample_df["total_importance"].describe())

print(sample_df.groupby("y_true")["total_importance"].mean())

total attribution per sample — summary
count    412.000000
mean       9.608748
std        9.377289
min        1.110172
25%        1.110172
50%        7.530877
75%       13.564264
max       36.667801
Name: total_importance, dtype: float64
y_true
0     9.949100
1    11.063497
2    22.562396
3    11.735937
4     2.947794
Name: total_importance, dtype: float64


In [16]:
class_names = ["Backdoor", "CommInj", "DoS", "Reconn", "normal"]


tp = y_true == y_pred
per_class_results = {}

# per attack feature importance
# focusing on correctly predicted classes
for c, name in enumerate(class_names):

    mask = tp & (y_true == c)

    if mask.sum().item() == 0:
        continue

    scores = feat_importance[mask].mean(dim=0)

    per_attack = pd.DataFrame({
        "feature": feature_names,
        "importance": scores.numpy()
    })

    per_attack = per_attack.sort_values("importance",ascending=False)
    per_class_results[name] = per_attack

    print(f"\n{name}: {mask.sum().item()} samples")
    display(per_attack.head(10))


Backdoor: 26 samples


,feature,importance
28,dst_app_bytes,2.260532
7,dst_bytes,2.126064
26,dst_ttl_mean,1.485172
16,dst_loss,1.345663
4,dst_pkts,0.835120
10,dst_load,0.622907
17,loss,0.571029
23,idle_mean,0.546788
35,dst_int_pkt_mean,0.389429
50,src_port_registered,0.127214



CommInj: 49 samples


,feature,importance
15,src_loss,3.961411
17,loss,1.940751
23,idle_mean,0.936485
27,src_app_bytes,0.643414
16,dst_loss,0.602038
13,dst_rate,0.598445
10,dst_load,0.565209
26,dst_ttl_mean,0.422338
4,dst_pkts,0.348240
56,dport_modbus,0.165538



DoS: 91 samples


,feature,importance
11,tot_load,5.209691
25,src_ttl_mean,2.555836
12,src_rate,2.553090
31,run_time_sum,2.029109
9,src_load,2.028589
14,tot_rate,1.916531
0,mean,1.626678
23,idle_mean,1.068303
34,src_int_pkt_mean,1.011142
32,src_jitter_mean,0.451289



Reconn: 29 samples


,feature,importance
2,num_dst_ports,7.178069
26,dst_ttl_mean,1.587903
23,idle_mean,1.326945
14,tot_rate,0.362871
39,proto_6,0.297519
58,dport_registered,0.257260
44,stos_0,0.217935
49,src_port_dynamic,0.196345
52,dport_dynamic,0.160881
9,src_load,0.134170



normal: 200 samples


,feature,importance
23,idle_mean,0.549272
1,num_src_ports,0.460178
26,dst_ttl_mean,0.296806
32,src_jitter_mean,0.247130
25,src_ttl_mean,0.211955
27,src_app_bytes,0.169888
15,src_loss,0.099278
34,src_int_pkt_mean,0.084387
35,dst_int_pkt_mean,0.083281
4,dst_pkts,0.072003
